# 收入邻域与多尺度编码LightGBM

负责人：A。这是中文教学参考，正式实现由成员理解后编写、执行并核对。

输入挂载：官方比赛数据、任务00输出的固定共享Dataset、原始EV数据集中的EV_Adoption_and_Range_Anxiety_Dataset.csv。无需挂载旧track代码包。CPU训练，四线程；机器等待另计。

公开方法适配与项目收入邻域实现。来源：Naji基础配方、多尺度编码与Zoom Zoom局部统计思想；完整历史函数在本文件展开。


- [比赛数据与规则](https://www.kaggle.com/competitions/playground-series-s6e9)
- [LightGBM论文](https://proceedings.neurips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html)
- [目标编码：内部交叉拟合与平滑](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html)
- [AUC定义](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

本教程生成时尚未执行Kaggle完整训练。历史分数是核对参照，不是本轮结果。阅读当前文档不意味着升级历史环境。

## 如何学习本文件

每次只运行一个单元，先用自己的话预测输出。`iloc`按位置取行，`loc`按标签取行；`to_numpy`去掉索引，之后必须保证位置对应。`assert`是验收条件，失败应查数据而非删除检查。`fit`从数据学习，`transform`使用已学习规则。

编程练习：修改一个小例子的输入并解释变化；正式配置保持历史定义。复杂特征组的整体增益不能归因于单一列。

## 读取官方数据

路径检查避免读错版本；对齐函数先检查ID集合，再恢复官方顺序。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import json
import gc
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import StratifiedKFold

INPUT = Path('/kaggle/input')
TARGET = 'Will_Buy_EV'
SEED = 42
N_SPLITS = 5

def unique_file(name):
    """从已挂载输入中定位唯一文件；多个版本时停止，防止静默读错。"""
    paths = list(INPUT.rglob(name))
    assert len(paths) == 1, f'Expected one {name}, found {paths}'
    return paths[0]

competition_dirs = [INPUT/'competitions/playground-series-s6e9', INPUT/'playground-series-s6e9']
available = [p for p in competition_dirs if (p/'train.csv').is_file()]
assert len(available) == 1, 'Attach the official competition data.'
DATA = available[0]
train = pd.read_csv(DATA/'train.csv')
test = pd.read_csv(DATA/'test.csv')
sample = pd.read_csv(DATA/'sample_submission.csv')
y = train[TARGET].map({'No':0, 'Yes':1})
assert y.notna().all() and set(y.unique()) == {0,1}
assert train.id.is_unique and test.id.is_unique
assert sample.columns.tolist() == ['id',TARGET] and sample.id.equals(test.id)
assert train.columns.drop(['id',TARGET]).tolist() == test.columns.drop('id').tolist()

def align_rows(frame, ids):
    """先检查一一对应，再按官方顺序排列；不能直接假设CSV行序相同。"""
    assert frame.id.is_unique and len(frame) == len(ids)
    assert set(frame.id) == set(ids)
    return frame.set_index('id').loc[ids].reset_index()

def current_versions():
    return {'lightgbm':lgb.__version__, 'sklearn':sklearn.__version__,
            'numpy':np.__version__, 'pandas':pd.__version__}

## 读取共同分组

共享fold决定训练与验证行，三人不能重新划分。

In [ ]:
fold_path = unique_file('shared_folds.csv')
shared = align_rows(pd.read_csv(fold_path), train.id)
assert np.array_equal(shared.target, y)
assert shared.fold.notna().all() and shared.fold.isin(range(5)).all()
assert set(shared.fold) == set(range(5))
fold_ids = shared.fold.to_numpy(dtype=int)
foundation_note = json.loads((fold_path.parent/'dataset_note.json').read_text())
display(shared.groupby('fold').agg(rows=('id','size'),positive_rate=('target','mean')))
print(current_versions())

## 核对环境和读取原始数据

表格比较历史与当前环境。若版本不同，先在Kaggle安装历史摘要记录的版本并重启会话，再从头运行；不要静默升级。原始数据用于统计特征，训练模型仍使用比赛训练行。

In [ ]:
expected_versions = foundation_note['historical_versions']['neighborhood']
normalized = {('sklearn' if k=='scikit_learn' else k):v for k,v in expected_versions.items()}
comparison = pd.DataFrame({'historical':normalized,'current':current_versions()})
display(comparison)
assert normalized, 'Historical environment metadata is missing.'
assert all(current_versions().get(k)==v for k,v in normalized.items() if k in current_versions()), 'Match historical library versions and restart the session.'
original_path = unique_file('EV_Adoption_and_Range_Anxiety_Dataset.csv')
original = pd.read_csv(original_path)
print(original.shape)

### 原始EV数据的身份与保留理由

[官方Data页面](https://www.kaggle.com/competitions/playground-series-s6e9/data)明确链接这份[Omkar Kadam发布的EV源数据](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety)，措辞是比赛训练/测试数据受其启发、分布接近但不相同。源数据本身是10,000行合成记录，不能当作真实调查。

历史强模型删除12列外部均值后OOF仅下降约0.00000515、公开榜显示持平；不能宣称外部均值已有可靠私人榜收益。本次保留是为了重建已验证的完整配置。

## 实现基础特征与收入邻域函数

函数工厂先准备原始数据均值，再返回每折特征函数。这里只定义函数，不训练模型。内部函数按顺序完成数字位→原始数据均值→频率→删除常数/重复信息→双平滑目标编码→邻域统计。`np.bincount`统计每箱人数和购买人数；平滑均值=(购买数+10×整体比例)/(人数+10)；`convolve`汇总左右相邻区间。斜率是右减左，曲率是中心减左右均值。8192/16384两种分箱提供不同尺度。

In [ ]:
def make_neighborhood_preparer(train, test, original):
    ID_COLUMN = "id"
    TARGET = "Will_Buy_EV"
    SEED = 42
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    from sklearn.preprocessing import TargetEncoder

    feature_columns = train.columns.drop(
        [ID_COLUMN, TARGET, "Number_of_Cars_Owned"]
    ).tolist()
    categorical_features = train[feature_columns].select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()
    numeric_features = [
        column for column in feature_columns
        if column not in categorical_features
    ]

    original_target = original[TARGET].map({"No": 0, "Yes": 1})
    assert original_target.notna().all()
    original_prior = original_target.mean()
    original_means = {
        column: original_target.groupby(original[column]).mean()
        for column in feature_columns
    }

    def build_public_features(raw):
        features = {column: raw[column] for column in feature_columns}
        digit_columns = []

        # 数字位提取保持原公式；浮点整除不可随意替换为四舍五入。
        for column in numeric_features:
            for position in range(-4, 4):
                name = f"{column}_digit{position}"
                features[name] = (
                    (raw[column] // (10.0 ** position)) % 10
                ).astype("int8")
                digit_columns.append(name)

        for column in feature_columns:
            features[f"{column}_org_mean"] = (
                raw[column].map(original_means[column]).fillna(original_prior)
            )

        encoding_columns = categorical_features.copy()
        for column in numeric_features + digit_columns:
            name = f"{column}_cat"
            features[name] = features[column].astype(str)
            encoding_columns.append(name)

        features["is_30k_spike"] = (
            raw["Annual_Income_USD"] == 30000
        ).astype("int8")
        features["is_env_hater"] = (
            raw["Environmental_Concern_Level"] == 1
        ).astype("int8")

        return pd.DataFrame(features, index=raw.index), encoding_columns


    def prepare_public_fold(train_idx, valid_idx):
        X_train, encoding_columns = build_public_features(train.iloc[train_idx])
        X_valid, _ = build_public_features(train.iloc[valid_idx])
        X_eval, _ = build_public_features(test)
        frames = [X_train, X_valid, X_eval]

        # 频率只用外层训练折；未知类别频率填0。
        for column in encoding_columns:
            frequency = X_train[column].value_counts(normalize=True)
            for frame in frames:
                frame[f"{column}_fe"] = frame[column].map(frequency).fillna(0.0)

        # 仅在训练折识别常数与完全相关列，保持列顺序和精确比较。
        constant_columns = X_train.columns[X_train.nunique() <= 1].tolist()
        numeric = X_train.select_dtypes(include="number").drop(
            columns=constant_columns, errors="ignore"
        )
        correlation = numeric.corr().abs()
        upper = correlation.where(
            np.triu(np.ones(correlation.shape, dtype=bool), k=1)
        )
        redundant_columns = upper.columns[(upper == 1.0).any()].tolist()
        dropped_columns = constant_columns + redundant_columns

        frames = [frame.drop(columns=dropped_columns) for frame in frames]
        encoding_columns = [
            column for column in encoding_columns
            if column not in dropped_columns
        ]
        X_train, X_valid, X_eval = frames

        # 两种平滑强度分别内部交叉拟合；验证和测试只transform。
        for smoothing, suffix in [("auto", "auto"), (10.0, "10")]:
            encoder = TargetEncoder(
                target_type="binary",
                smooth=smoothing,
                cv=5,
                shuffle=True,
                random_state=SEED,
            )
            encoded = [
                encoder.fit_transform(
                    X_train[encoding_columns], y.iloc[train_idx]
                ),
                encoder.transform(X_valid[encoding_columns]),
                encoder.transform(X_eval[encoding_columns]),
            ]
            columns = [
                f"{column}_TE_{suffix}" for column in encoding_columns
            ]
            for index, values in enumerate(encoded):
                frames[index] = pd.concat([
                    frames[index],
                    pd.DataFrame(
                        values.astype("float32"),
                        columns=columns,
                        index=frames[index].index,
                    ),
                ], axis=1)

        return [
            frame.drop(columns=encoding_columns)
            for frame in frames
        ]

    def income_neighborhood_table(bin_ids, labels, n_bins, smooth=10.0):
        # bincount统计每个收入区间人数；带weights时统计正例总数。
        counts = np.bincount(bin_ids, minlength=n_bins)
        totals = np.bincount(bin_ids, weights=labels, minlength=n_bins)
        prior = labels.mean()
        # 平滑：增加smooth个按整体比例分布的虚拟样本。
        center = (totals + smooth * prior) / (counts + smooth)
        left = np.r_[prior, center[:-1]]
        right = np.r_[center[1:], prior]
        # 高斯核让相邻区间贡献较小权重；convolve汇总左右邻居。
        kernel = np.exp(-0.5 * (np.arange(-1, 2) / 0.8) ** 2)
        local_counts = np.convolve(counts, kernel, mode="same")
        local_totals = np.convolve(totals, kernel, mode="same")
        local_mean = (
            (local_totals + smooth * kernel.sum() * prior)
            / (local_counts + smooth * kernel.sum())
        )
        return np.column_stack([
            center, left, right, local_mean, right - left,
            center - (left + right) / 2, np.log1p(counts),
        ])

    def prepare_income_neighborhoods(train_idx, valid_idx):
        income_train = train["Annual_Income_USD"].to_numpy()[train_idx]
        income_valid = train["Annual_Income_USD"].to_numpy()[valid_idx]
        income_test = test["Annual_Income_USD"].to_numpy()
        labels = np.asarray(y)[train_idx]
        inner_folds = list(StratifiedKFold(
            n_splits=5, shuffle=True, random_state=17
        ).split(income_train, labels))
        blocks = [[], [], []]
        columns = []
        stat_names = [
            "Rate", "Left_Rate", "Right_Rate", "Local_Mean",
            "Slope", "Curvature", "Log_Support",
        ]
        for n_bins in (8192, 16384):
            edges = np.linspace(
                income_train.min(), income_train.max(), n_bins + 1
            )
            train_bins, valid_bins, test_bins = [
                np.searchsorted(edges[1:-1], values, side="right")
                for values in (income_train, income_valid, income_test)
            ]
            # 空数组按内部留出行回填，避免该行标签直接参与自身编码。
            train_features = np.empty((len(train_idx), 7), dtype=np.float32)
            for fit_idx, holdout_idx in inner_folds:
                table = income_neighborhood_table(
                    train_bins[fit_idx], labels[fit_idx], n_bins
                )
                train_features[holdout_idx] = table[train_bins[holdout_idx]]
            table = income_neighborhood_table(train_bins, labels, n_bins)
            for destination, values in zip(
                blocks, (train_features, table[valid_bins], table[test_bins])
            ):
                destination.append(values)
            columns.extend(f"Income_{n_bins}_{name}" for name in stat_names)
            if n_bins == 8192:
                for destination, bin_ids in zip(
                    blocks, (train_bins, valid_bins, test_bins)
                ):
                    destination.append((bin_ids / (n_bins - 1))[:, None])
                columns.append("Income_Bin_Position")
        return tuple(
            pd.DataFrame(np.column_stack(parts), columns=columns).astype("float32")
            for parts in blocks
        )

    def prepare_fold(train_idx, valid_idx):
        base_frames = prepare_public_fold(train_idx, valid_idx)
        extra_frames = prepare_income_neighborhoods(train_idx, valid_idx)
        combined = []
        for base, extra in zip(base_frames, extra_frames):
            assert len(base) == len(extra)
            frame = pd.concat([
                base.reset_index(drop=True), extra.reset_index(drop=True)
            ], axis=1)
            assert frame.columns.is_unique
            assert np.isfinite(frame.to_numpy(dtype=float)).all()
            combined.append(frame)
        return combined

    return prepare_fold

### 对照上方代码逐句理解

行号从上方代码单元第一行起计。重复操作也列出，方便逐行定位；跨行调用请连同后续参数一起阅读。

| 行 | 代码定位 | 中文解释 |
|---|---|---|
| 5 | `y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 37 | `).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 42 | `raw[column].map(original_means[column]).fillna(original_prior)` | 为未匹配或缺失项提供明确回退值；均值特征通常回退整体购买比例。 |
| 48 | `features[name] = features[column].astype(str)` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 53 | `).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 56 | `).astype("int8")` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 69 | `frequency = X_train[column].value_counts(normalize=True)` | 从当前训练部分计算频率；normalize=True返回比例，否则返回次数。 |
| 71 | `frame[f"{column}_fe"] = frame[column].map(frequency).fillna(0.0)` | 为未匹配或缺失项提供明确回退值；均值特征通常回退整体购买比例。 |
| 80 | `np.triu(np.ones(correlation.shape, dtype=bool), k=1)` | 只检查相关矩阵上三角，删除后出现的重复信息列，保留之前的列顺序。 |
| 102 | `encoder.fit_transform(` | 训练行使用编码器内部交叉拟合结果，不能替换成fit后对训练集transform。 |
| 105 | `encoder.transform(X_valid[encoding_columns]),` | 使用外层训练数据已学到的映射，不输入验证或测试标签。 |
| 106 | `encoder.transform(X_eval[encoding_columns]),` | 使用外层训练数据已学到的映射，不输入验证或测试标签。 |
| 112 | `frames[index] = pd.concat([` | 按列合并特征；索引必须一致，reset_index用于避免错位产生缺失值。 |
| 115 | `values.astype("float32"),` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 128 | `counts = np.bincount(bin_ids, minlength=n_bins)` | 按整数分组编号累计人数；weights指定时累计对应标签，得到购买人数。 |
| 129 | `totals = np.bincount(bin_ids, weights=labels, minlength=n_bins)` | 按整数分组编号累计人数；weights指定时累计对应标签，得到购买人数。 |
| 137 | `local_counts = np.convolve(counts, kernel, mode="same")` | 把每个收入箱及左右邻箱按核权重汇总，得到更稳定的局部统计。 |
| 138 | `local_totals = np.convolve(totals, kernel, mode="same")` | 把每个收入箱及左右邻箱按核权重汇总，得到更稳定的局部统计。 |
| 143 | `return np.column_stack([` | 把等长数组并排组成二维特征矩阵，顺序与列名列表对应。 |
| 163 | `edges = np.linspace(` | 在训练收入最小值与最大值之间建立等宽边界。 |
| 167 | `np.searchsorted(edges[1:-1], values, side="right")` | 按已学习的边界把数值映射为分组编号；right决定恰好位于边界时的归属。 |
| 190 | `pd.DataFrame(np.column_stack(parts), columns=columns).astype("float32")` | 把等长数组并排组成二维特征矩阵，顺序与列名列表对应。 |
| 200 | `frame = pd.concat([` | 按列合并特征；索引必须一致，reset_index用于避免错位产生缺失值。 |

## 追加四种分组的购买统计

例如收入12345分别映射到12345、123、12；粗分组样本多但分辨率低。训练编码使用fit_transform内部交叉拟合，验证和测试仅transform。smooth=100将稀疏组向整体比例收缩。

In [ ]:
prepare_neighborhood = make_neighborhood_preparer(train,test,original)
def prepare_fold(training_index,validation_index):
    frames = prepare_neighborhood(training_index,validation_index)
    keys = [pd.DataFrame({
        'Income_Integer':np.floor(part.Annual_Income_USD).astype('int64'),
        'Income_100':np.floor(part.Annual_Income_USD/100).astype('int64'),
        'Income_1000':np.floor(part.Annual_Income_USD/1000).astype('int64'),
        'Commute_Integer':np.floor(part.Daily_Commute_km).astype('int64')
    },index=part.index) for part in [train.iloc[training_index],train.iloc[validation_index],test]]
    encoder = TargetEncoder(target_type='binary',smooth=100.0,cv=5,shuffle=True,random_state=42)
    values = [encoder.fit_transform(keys[0],y.iloc[training_index]),encoder.transform(keys[1]),encoder.transform(keys[2])]
    for frame,encoded in zip(frames,values):
        frame[[f'{col}_TE100' for col in keys[0]]] = encoded.astype('float32')
        assert np.isfinite(frame.to_numpy(dtype=float)).all()
    return frames

### 对照上方代码逐句理解

行号从上方代码单元第一行起计。重复操作也列出，方便逐行定位；跨行调用请连同后续参数一起阅读。

| 行 | 代码定位 | 中文解释 |
|---|---|---|
| 5 | `'Income_Integer':np.floor(part.Annual_Income_USD).astype('int64'),` | 向下取整建立分组；输入除以100或1000时表示更粗的收入区间。 |
| 6 | `'Income_100':np.floor(part.Annual_Income_USD/100).astype('int64'),` | 向下取整建立分组；输入除以100或1000时表示更粗的收入区间。 |
| 7 | `'Income_1000':np.floor(part.Annual_Income_USD/1000).astype('int64'),` | 向下取整建立分组；输入除以100或1000时表示更粗的收入区间。 |
| 8 | `'Commute_Integer':np.floor(part.Daily_Commute_km).astype('int64')` | 向下取整建立分组；输入除以100或1000时表示更粗的收入区间。 |
| 11 | `values = [encoder.fit_transform(keys[0],y.iloc[training_index]),encoder.transform(keys[1]),encoder.transform(keys[2])]` | 训练行使用编码器内部交叉拟合结果，不能替换成fit后对训练集transform。 |
| 13 | `frame[[f'{col}_TE100' for col in keys[0]]] = encoded.astype('float32')` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |

## 固定历史训练参数

学习率0.005需要更多树。subsample_freq=0表示历史配置没有启用按轮行采样，即使subsample写了0.8，也不能擅自改为1。列采样使列顺序影响复现。

In [ ]:
model_params = {'objective': 'binary', 'metric': 'auc', 'n_estimators': 40000, 'learning_rate': 0.005, 'max_depth': 5, 'num_leaves': 32, 'min_child_samples': 10, 'subsample': 0.8, 'subsample_freq': 0, 'colsample_bytree': 0.3, 'reg_alpha': 0.071, 'reg_lambda': 2.0, 'max_bin': 1024, 'random_state': 42, 'feature_pre_filter': False, 'n_jobs': 4, 'verbosity': -1}
patience, expected_count = 700, 159

## 五折训练并回填预测

这是主要耗时单元。每折留出五分之一用于评价；早停也使用这个验证集，因此OOF属于开发评价。predict_proba的[:,1]取购买概率。五次覆盖完成后每行coverage应为1。不要为了整理日志重训。

In [ ]:
run_name = 'neighborhood'
method_sources = ['https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94587-lb-0-94612?scriptVersionId=346904855', 'https://www.kaggle.com/code/jazivxt/single-model-zoom-zoom', 'https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637']

candidate_oof = np.full(len(train),np.nan)  # 没有预测的行保持NaN，便于发现漏填。
candidate_test = np.zeros(len(test))
coverage = np.zeros(len(train),dtype=np.uint8)
records, fold_features = [], {}
started = perf_counter()
for fold in range(N_SPLITS):
    training_index = np.flatnonzero(fold_ids != fold)
    validation_index = np.flatnonzero(fold_ids == fold)
    X_train, X_valid, X_test = prepare_fold(training_index,validation_index)
    assert X_train.columns.equals(X_valid.columns) and X_train.columns.equals(X_test.columns)
    if expected_count is not None:
        assert X_train.shape[1] == expected_count, (fold,X_train.shape)
        historical_columns = foundation_note.get('historical_features',{}).get(run_name,{}).get(str(fold))
        if historical_columns is not None:
            assert X_train.columns.tolist() == historical_columns, 'Historical feature order differs.'
    fold_features[str(fold)] = X_train.columns.tolist()
    if run_name != 'baseline':
        assert model_params == foundation_note['historical_params'][run_name], 'Historical parameters differ.'
    model = lgb.LGBMClassifier(**model_params)  # 每个外层fold创建全新模型。
    fold_started = perf_counter()
    model.fit(X_train,y.iloc[training_index],eval_set=[(X_valid,y.iloc[validation_index])],
              eval_metric='auc',callbacks=[lgb.early_stopping(patience,first_metric_only=True,verbose=False),lgb.log_evaluation(1000)])
    probability = model.predict_proba(X_valid,num_iteration=model.best_iteration_)[:,1]
    test_probability = model.predict_proba(X_test,num_iteration=model.best_iteration_)[:,1]
    assert np.isfinite(probability).all() and np.isfinite(test_probability).all()
    candidate_oof[validation_index] = probability  # 回填官方训练行的位置。
    coverage[validation_index] += 1
    candidate_test += test_probability/N_SPLITS  # 在概率空间平均，暂不排名。
    records.append({'fold':fold,'auc':float(roc_auc_score(y.iloc[validation_index],probability)),
                    'features':X_train.shape[1],'best_iteration':int(model.best_iteration_),
                    'fit_seconds':perf_counter()-fold_started})
    print(records[-1])
    del model,X_train,X_valid,X_test
    gc.collect()
assert (coverage==1).all() and np.isfinite(candidate_oof).all()
assert ((candidate_oof>=0)&(candidate_oof<=1)).all()
assert ((candidate_test>=0)&(candidate_test<=1)).all()
elapsed = perf_counter()-started
display(pd.DataFrame(records))
print('Overall OOF AUC:',roc_auc_score(y,candidate_oof))

## 保存并交接真实结果

只保存完整OOF、测试预测和摘要。输出目录已经存在时停止，先保存上次结果后重开会话。把整个目录保存为Notebook输出，告诉融合负责人挂载。

In [ ]:
output = Path('/kaggle/working')/run_name
output.mkdir(exist_ok=False)
oof = pd.DataFrame({'id':train.id,'target':y,'fold':fold_ids,'prediction':candidate_oof})
submission = sample.copy()
submission[TARGET] = candidate_test
auc = float(roc_auc_score(y,candidate_oof))
report = (f'The {run_name} model was trained on five shared folds. Overall OOF ROC AUC was {auc:.9f}. '
          'Test probabilities were averaged across five models. All competition-derived supervised '
          'preprocessing was fitted within outer training folds. These are development results; '
          'leaderboard performance for this run remains unverified.')
summary = {'run_name':run_name,'model_params':model_params,'stopping_rounds':patience,
           'fold_features':fold_features,'fold_metrics':records,'oof_auc':auc,'elapsed_seconds':elapsed,
           'versions':current_versions(),'fold_source':str(fold_path),'method_sources':method_sources,
           'test_aggregation':'mean probabilities across five folds','public_score':None,'report_summary':report}
oof.to_csv(output/'oof_predictions.csv',index=False)
submission.to_csv(output/'submission.csv',index=False)
(output/'run_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')
print(report)
print('Saved:',output)

## 中文结果解析与理解检查

逐折AUC比较必须使用相同fold。整体OOF AUC和五折AUC的平均不是同一个量。两个完整方案同时改变多组特征，不能宣称某一列造成全部提升。

请回答：为什么测试集不参与早停？为什么目标编码训练行不能直接使用全训练折groupby均值？为什么相同特征数量仍可能有不同列序？请画出一行样本从原始数据到验证预测经过的步骤。

运行后用实际输出写中文观察；摘要已自动生成英文Report Summary。历史参照：收入邻域模型0.946046626，混合特征模型0.946129123；未训练前不能把这些数写成本次结果。